# Candidate Detection with IR-MAD

IR-MAD based optical and SAR change fusion, then SLIC + hysteresis + blob ranking.

## Candidate Detection Pipeline (Current Behavior)

This notebook scans incident rasters already uploaded to Hugging Face and produces ranked landslide candidate chips.

### 1) What Inputs It Requires Per Incident
The incident is processed only when these raw files exist in `raw_images/raw_incidents/incident_{id}/`:
- `incident_{id}_after.tif`
- one of `incident_{id}_planet_before.tif` or `incident_{id}_gee_before.tif`
- `incident_{id}_slope.tif`
- `incident_{id}_sar_pre.tif`
- `incident_{id}_sar_post.tif`

Incidents that already have any candidate outputs under `candidates/incident_{id}/` are skipped.

### 2) Optical Handling (Updated for Quarterly Visual Mosaics)
Planet optical imagery is now treated as **RGB-first**:
- If a 4th band looks like alpha/mask (visual RGBA), the notebook uses RGB bands only.
- If the raster looks analytic-style, it still maps to RGB channels consistently.
- The optical model operates on RGB change evidence.

Important: NDVI/NIR vegetation-loss fusion is currently disabled in this workflow (`have_veg = False`) to avoid misusing visual alpha as NIR.

### 3) Common Analysis Grid
Before/after optical, SAR, and slope are reprojected onto one common grid:
- CRS: `EPSG:4326`
- Approx. resolution: `GEE_SCALE_M` (10 m)

This ensures all modalities align pixel-to-pixel for fusion and segmentation.

### 4) Change Features and Fusion
Per incident, the notebook computes:
- Optical change confidence via IR-MAD on RGB channels.
- SAR change confidence via IR-MAD on VV/VH.
- Slope mask (`MIN_SLOPE_DEG`) to constrain detection to steep terrain.

Fusion is geometric mean over available modalities (optical and SAR), then multiplied by slope mask.

### 5) Candidate Generation
The fused map is converted to candidates by:
- SLIC superpixel segmentation (target physical segment size).
- Per-segment significance test (Bonferroni-controlled via `FWER_ALPHA`).
- Hysteresis thresholding from per-incident percentiles (`HIGH_CONF_PERCENTILE`, `LOW_CONF_PERCENTILE`).
- Morphology and blob filters (`MIN_BLOB_AREA_M2`, `MAX_BLOB_AREA_M2`, `MAX_ELONGATION`).
- Ranking by `score = area_m2 * severity`, capped at `MAX_CANDIDATES_PER_INCIDENT`.

### 6) Outputs Written to Hugging Face
For each kept candidate, the notebook writes:
- `before` chip (native source resolution)
- `after` chip (native source resolution)
- `slope` chip (analysis grid)
- `mask` chip (analysis grid)

Outputs go to:
- `candidates/incident_{id}/candidate_{k}/...`
- `candidates/candidate_status.csv`
- `candidates/candidate_metadata.csv`

### 7) Concurrency and Uploads
- Incident processing runs in parallel (`MAX_WORKERS`).
- Candidate chip uploads are committed in batches (`UPLOAD_BATCH_SIZE`) using Hugging Face commit operations.

In [ ]:
import os
import re
import shutil
import numpy as np
import pandas as pd
import rasterio
from concurrent.futures import ThreadPoolExecutor, as_completed
from rasterio.warp import calculate_default_transform, reproject, transform_bounds, Resampling
from rasterio.windows import Window, transform as window_transform
from scipy.stats import chi2, norm
from skimage.segmentation import slic
from skimage.filters import apply_hysteresis_threshold
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
HF_RAW_ROOT = 'raw_images/raw_incidents'
HF_CAND_ROOT = 'candidates'
HIGH_CONF_PERCENTILE = 99.7   # per-incident percentile (within slope-masked terrain) used as the hysteresis 'seed' threshold
LOW_CONF_PERCENTILE = 93.0    # per-incident percentile used as the hysteresis 'grow' threshold
MIN_SLOPE_DEG = 20
# Bonferroni-corrected false-positive rate for the per-segment significance gate below -
# without this, HIGH_CONF_PERCENTILE alone always "detects" something in every incident,
# since the max of many noisy per-segment change scores drifts toward 1 by chance alone
# as the number of segments grows, even with zero real change anywhere in the AOI.
FWER_ALPHA = 0.05
MAX_ITERS = 20
MAX_CANDIDATES_PER_INCIDENT = 100
MIN_BLOB_AREA_M2 = 3000
MAX_BLOB_AREA_M2 = 2000000
MAX_ELONGATION = 5.0
CHIP_PAD_M = 300   # padding (meters) added around each candidate's bbox for the exported chips
UPLOAD_BATCH_SIZE = 25   # chip files per batched HF commit
MAX_WORKERS = 4    # number of incidents processed concurrently

# Common analysis grid: everything (before/after/SAR/slope) is reprojected onto this
# single grid so pixels correspond to the same ground location. Target resolution
# matches GEE's native Sentinel-2/SRTM sampling (~10 m) per the pipeline spec.
GEE_SCALE_M = 10
TARGET_CRS = 'EPSG:4326'
TARGET_SUPERPIXEL_M = 30   # desired physical superpixel footprint, drives SLIC segment count
M_PER_DEG_LAT = 111320.0   # meters per degree of latitude, ~constant everywhere

def norm01(x):
    a = np.nanmin(x); b = np.nanmax(x)
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    return ((x-a)/(b-a)).astype(np.float32)

def robust01(x, lo_pct=1, hi_pct=99):
    # 1st/99th percentile clip before min-max, so a few extreme outlier pixels can't
    # compress the rest of the distribution to near-zero (unlike norm01's plain min/max).
    lo, hi = np.percentile(x, [lo_pct, hi_pct])
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return np.clip((x - lo) / (hi - lo), 0, 1).astype(np.float32)

def wmean_cov(X, w):
    w = w / np.sum(w)
    m = np.sum(X * w[:, None], axis=0)
    Xc = X - m
    C = (Xc * w[:, None]).T @ Xc
    return m, C

def cca_axes(Sxx, Syy, Sxy, reg=1e-6):
    p = Sxx.shape[0]
    q = Syy.shape[0]
    # Cholesky whitening converts to symmetric problem; avoids eigh on non-symmetric M
    Lx = np.linalg.cholesky(Sxx + reg * np.eye(p))
    Ly = np.linalg.cholesky(Syy + reg * np.eye(q))
    invLx = np.linalg.inv(Lx)
    invLy = np.linalg.inv(Ly)
    K = invLx @ Sxy @ invLy.T
    U, _, Vt = np.linalg.svd(K, full_matrices=False)
    A = invLx.T @ U
    B = invLy.T @ Vt.T
    return A, B

def irmad(X, Y, max_iters=MAX_ITERS, tol=1e-3):
    # NOTE: `chi` is asymptotically chi-square distributed under the null hypothesis of
    # NO change, so a LARGE chi means a pixel is unlikely to be unchanged (i.e. likely
    # changed). `no_change_p` (the survival function / p-value) is therefore HIGH for
    # stable/unchanged pixels and is what IRMAD's reweighting scheme wants (unchanged
    # pixels get more weight when re-estimating the no-change statistics each iteration).
    # The function's return value, however, must be a CHANGE confidence (high = likely
    # changed) since callers use it directly to build the fused change-detection mask -
    # returning `no_change_p` here would flag the stable background as "candidates" and
    # miss real change areas, which previously caused near-zero true detections.
    n, p = X.shape
    w = np.ones(n, dtype=np.float64)
    prev = None
    for _ in range(max_iters):
        mx, Sxx = wmean_cov(X, w)
        my, Syy = wmean_cov(Y, w)
        Xc = X - mx
        Yc = Y - my
        wn = w / np.sum(w)
        Sxy = (Xc * wn[:, None]).T @ Yc
        A, B = cca_axes(Sxx, Syy, Sxy)
        U = Xc @ A[:, :p]
        V = Yc @ B[:, :p]
        M = U - V
        s = np.std(M, axis=0) + 1e-6
        chi = np.sum((M / s[None, :]) ** 2, axis=1)
        no_change_p = 1.0 - chi2.cdf(chi, df=p)
        if prev is not None and np.mean(np.abs(no_change_p - prev)) < tol:
            break
        prev = no_change_p
        w = np.clip(no_change_p, 1e-6, 1.0)
    change_conf = chi2.cdf(chi, df=p)
    return chi, change_conf

def hf_fetch(token, incident_id, fn):
    return hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION, filename=f'{HF_RAW_ROOT}/incident_{incident_id}/{fn}', token=token)

def build_target_grid(ref_path, scale_m=GEE_SCALE_M, dst_crs=TARGET_CRS):
    with rasterio.open(ref_path) as src:
        # Meters-per-degree-longitude shrinks with cos(latitude); using a uniform
        # 111320 m/deg for both axes over-estimates longitude resolution away from the
        # equator (~10-14% at Nepal's ~26-30N). Compute the AOI's mean latitude and use
        # it to scale the longitude (x) resolution separately from latitude (y).
        lon_min, lat_min, lon_max, lat_max = transform_bounds(src.crs, dst_crs, *src.bounds)
        mean_lat_rad = np.deg2rad((lat_min + lat_max) / 2.0)
        m_per_deg_lon = M_PER_DEG_LAT * max(np.cos(mean_lat_rad), 1e-6)
        res_deg_lon = scale_m / m_per_deg_lon
        res_deg_lat = scale_m / M_PER_DEG_LAT
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=(res_deg_lon, res_deg_lat))
    return transform, int(width), int(height), mean_lat_rad

def reproject_bands(path, band_indexes, dst_transform, dst_crs, dst_shape, resampling=Resampling.bilinear):
    h, w = dst_shape
    out = np.zeros((len(band_indexes), h, w), dtype=np.float32)
    with rasterio.open(path) as src:
        for i, band_idx in enumerate(band_indexes):
            reproject(
                source=rasterio.band(src, band_idx),
                destination=out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=resampling,
            )
    return out

def _looks_like_alpha_band(band):
    vals = band[np.isfinite(band)]
    if vals.size < 100:
        return False
    p1, p99 = np.percentile(vals, [1, 99])
    dyn = p99 - p1
    uniq = len(np.unique(vals))
    # Alpha-like bands are usually near-binary masks with tiny dynamic range.
    return (dyn < 2.0) or (uniq <= 8)


def select_rgb_band_indexes(path):
    with rasterio.open(path) as src:
        n = src.count
        if n >= 4:
            b4 = src.read(4)
            if _looks_like_alpha_band(b4):
                # Visual mosaic layout: R,G,B,Alpha
                return [1, 2, 3], False
            # Analytic/S2 layout: B,G,R,NIR
            return [3, 2, 1], True
        if n >= 3:
            # Assume RGB order when only 3 bands are present.
            return [1, 2, 3], False
        raise ValueError(f'Optical raster has too few bands ({n}) in {path}')


def load_optical(before_path, after_path, dst_transform, dst_crs, dst_shape):
    before_rgb_idx, before_has_nir = select_rgb_band_indexes(before_path)
    after_rgb_idx, after_has_nir = select_rgb_band_indexes(after_path)
    n_bands = min(len(before_rgb_idx), len(after_rgb_idx), 3)
    before_idx = before_rgb_idx[:n_bands]
    after_idx = after_rgb_idx[:n_bands]

    before = reproject_bands(before_path, before_idx, dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    after = reproject_bands(after_path, after_idx, dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    # Mark pixels where both rasters have actual data in all loaded bands.
    # rasterio.warp.reproject fills no-data regions with 0, so a pixel that is
    # zero across all bands in either image is treated as no-data and excluded
    # from IR-MAD fitting.
    valid = np.all(before > 0, axis=0) & np.all(after > 0, axis=0)
    meta = {
        'before_rgb_band_indexes': before_idx,
        'after_rgb_band_indexes': after_idx,
        'has_nir_pair': bool(before_has_nir and after_has_nir),
    }
    return before.astype(np.float32), after.astype(np.float32), valid, meta

def crop_native(src_path, band_indexes, lonlat_bbox):
    """Crop src_path to the given (lon_min, lat_min, lon_max, lat_max) bbox,
    returning (data, chip_transform, crs) at the file's native resolution.
    Used to export before/after chips in original Planet scale rather than the
    10 m analysis grid.
    """
    lon_min, lat_min, lon_max, lat_max = lonlat_bbox
    with rasterio.open(src_path) as src:
        sl, sb, sr, st = transform_bounds('EPSG:4326', src.crs, lon_min, lat_min, lon_max, lat_max)
        win = src.window(sl, sb, sr, st)
        col_off = max(0, int(np.floor(win.col_off)))
        row_off = max(0, int(np.floor(win.row_off)))
        col_end = min(src.width, int(np.ceil(win.col_off + win.width)))
        row_end = min(src.height, int(np.ceil(win.row_off + win.height)))
        win_int = Window(col_off=col_off, row_off=row_off,
                         width=max(1, col_end - col_off),
                         height=max(1, row_end - row_off))
        data = src.read(band_indexes, window=win_int).astype(np.float32)
        chip_tf = src.window_transform(win_int)
        crs = src.crs
    return data, chip_tf, crs

def load_slope(path, dst_transform, dst_crs, dst_shape):
    return reproject_bands(path, [1], dst_transform, dst_crs, dst_shape, Resampling.bilinear)[0]

def load_sar(pre_path, post_path, dst_transform, dst_crs, dst_shape):
    pre = reproject_bands(pre_path, [1, 2], dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    post = reproject_bands(post_path, [1, 2], dst_transform, dst_crs, dst_shape, Resampling.bilinear)
    return pre, post

def write_chip(path, arr, transform, crs):
    data = arr[None, ...] if arr.ndim == 2 else arr
    meta = {
        'driver': 'GTiff',
        'dtype': 'float32',
        'count': data.shape[0],
        'height': data.shape[1],
        'width': data.shape[2],
        'crs': crs,
        'transform': transform,
    }
    with rasterio.open(path, 'w', **meta) as dst:
        dst.write(data.astype(np.float32))

secrets = UserSecretsClient()
hf_token = secrets.get_secret('huggingface_token')
api = HfApi(token=hf_token)

# Discover which incidents actually have raw imagery uploaded to HF, and which of those
# already have candidate outputs - scan the HF repo directly instead of looping over
# every incident in the source CSV (incident_download.ipynb uploads incrementally and
# out of CSV-row order, so the CSV no longer reflects what's actually ready to process).
try:
    all_repo_files = set(api.list_repo_files(HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION))
except Exception:
    all_repo_files = set()

raw_file_re = re.compile(rf'^{re.escape(HF_RAW_ROOT)}/incident_(\d+)/(.+)$')
raw_files_by_incident = {}
for f in all_repo_files:
    m = raw_file_re.match(f)
    if m:
        raw_files_by_incident.setdefault(int(m.group(1)), set()).add(m.group(2))

MANDATORY_RAW_SUFFIXES = {'after.tif', 'slope.tif', 'sar_pre.tif', 'sar_post.tif'}

def has_mandatory_raw(inc_id, files):
    prefix = f'incident_{inc_id}_'
    suffixes = {fn[len(prefix):] for fn in files if fn.startswith(prefix)}
    if not MANDATORY_RAW_SUFFIXES.issubset(suffixes):
        return False
    return 'planet_before.tif' in suffixes or 'gee_before.tif' in suffixes

incident_ids = sorted(inc_id for inc_id, files in raw_files_by_incident.items() if has_mandatory_raw(inc_id, files))
print(f'Incidents with complete raw imagery on HF: {len(incident_ids)}')

# Skip incidents that already have candidate outputs on HF (idempotent re-runs).
existing_cand_files = {f for f in all_repo_files if f.startswith(f'{HF_CAND_ROOT}/')}

pending_ops = []    # list[CommitOperationAdd] waiting for the next batched HF commit
pending_meta = []   # list[(inc_id, local_dir)] describing what pending_ops holds

def flush_pending():
    global pending_ops, pending_meta
    if not pending_ops:
        return
    n_incidents = len(pending_meta)
    try:
        api.create_commit(
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            operations=pending_ops,
            commit_message=f'Add candidate chips for {n_incidents} incident(s)',
        )
        for inc_id, _ in pending_meta:
            print(f'Uploaded candidate chips for incident_{inc_id} (batch flush of {len(pending_ops)} files / {n_incidents} incidents)')
    except Exception as e:
        for inc_id, _ in pending_meta:
            print(f'Batch upload failed for incident_{inc_id}: {e}')
    finally:
        for _, local_dir in pending_meta:
            shutil.rmtree(local_dir, ignore_errors=True)
        pending_ops = []
        pending_meta = []

def process_incident(inc_id):
    after_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_after.tif')
    slope_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_slope.tif')
    try:
        before_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_planet_before.tif')
    except Exception:
        before_path = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_gee_before.tif')
    sar_pre = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_sar_pre.tif')
    sar_post = hf_fetch(hf_token, inc_id, f'incident_{inc_id}_sar_post.tif')

    dst_transform, gw, gh, mean_lat_rad = build_target_grid(before_path)
    dst_shape = (gh, gw)
    h, w = dst_shape

    before_rgb, after_rgb, valid, optical_meta = load_optical(before_path, after_path, dst_transform, TARGET_CRS, dst_shape)
    slope = load_slope(slope_path, dst_transform, TARGET_CRS, dst_shape)
    slope_mask = slope >= MIN_SLOPE_DEG

    n_opt_bands = before_rgb.shape[0]
    Xo = np.moveaxis(before_rgb, 0, -1).reshape(-1, n_opt_bands)
    Yo = np.moveaxis(after_rgb, 0, -1).reshape(-1, n_opt_bands)
    pm = valid.reshape(-1)
    conf_opt = np.zeros(h * w, dtype=np.float32)
    chi_opt_full = np.zeros(h * w, dtype=np.float64)
    df_opt_full = np.zeros(h * w, dtype=np.float64)
    if np.sum(pm) > 100:
        chi_o, conf_o = irmad(Xo[pm], Yo[pm])
        conf_opt[pm] = conf_o
        chi_opt_full[pm] = chi_o
        df_opt_full[pm] = n_opt_bands

    spre, spost = load_sar(sar_pre, sar_post, dst_transform, TARGET_CRS, dst_shape)
    Xs = np.moveaxis(spre, 0, -1).reshape(-1, 2)
    Ys = np.moveaxis(spost, 0, -1).reshape(-1, 2)
    # Build a SAR-specific validity mask (non-zero in all bands for both pre and post),
    # separate from the optical pm so SAR no-data pixels (rasterio.warp fills with 0
    # outside the scene extent) are not fed into the SAR weighted regression.
    # Use != 0 (not > 0) so dB-scale SAR (negative valid values) is not excluded;
    # rasterio.warp.reproject fills no-data regions with exactly 0, which is the
    # sentinel we need to reject regardless of sign convention.
    pm_sar = (np.all(spre != 0, axis=0) & np.all(spost != 0, axis=0)).reshape(-1)
    conf_sar = np.zeros(h * w, dtype=np.float32)
    chi_sar_full = np.zeros(h * w, dtype=np.float64)
    df_sar_full = np.zeros(h * w, dtype=np.float64)
    if np.sum(pm_sar) > 100:
        chi_s, conf_s = irmad(Xs[pm_sar], Ys[pm_sar])
        conf_sar[pm_sar] = conf_s
        chi_sar_full[pm_sar] = chi_s
        df_sar_full[pm_sar] = 2

    conf_opt2 = conf_opt.reshape(h, w)
    conf_sar2 = conf_sar.reshape(h, w)
    # Each modality is only trustworthy once it has enough valid pixels (see the
    # `np.sum(pm...) > 100` guards above) - only fuse whichever modalities are usable
    # rather than assuming all are always present, otherwise a sparse/unavailable
    # modality's all-zero confidence silently zeroes out the geometric-mean fusion.
    have_opt = np.sum(pm) > 100
    have_sar = np.sum(pm_sar) > 100

    # Quarterly visual mosaics are RGB-only in this plan, so keep optical fusion
    # strictly on RGB change evidence and skip NDVI/NIR-based vegetation-loss fusion.
    have_veg = False

    confs = []
    if have_opt:
        confs.append(np.clip(conf_opt2, 0, 1))
    if have_sar:
        confs.append(np.clip(conf_sar2, 0, 1))
    if have_veg:
        confs.append(veg_loss_conf)
    if confs:
        fused = np.power(np.prod(np.stack(confs, axis=0), axis=0), 1.0 / len(confs)).astype(np.float32)
    else:
        fused = np.zeros((h, w), dtype=np.float32)
    fused *= slope_mask.astype(np.float32)

    # Combined per-pixel change-evidence chi statistic + its degrees of freedom (sum of
    # whichever of optical/SAR are valid at that pixel) - used below to test whether a
    # segment's aggregate evidence is statistically distinguishable from pure noise.
    chi_comb_full = (chi_opt_full + chi_sar_full).reshape(h, w)
    df_comb_full = (df_opt_full + df_sar_full).reshape(h, w)

    feat = np.concatenate([fused[..., None], np.stack([norm01(after_rgb[i]) for i in range(n_opt_bands)], axis=-1), norm01(slope)[..., None]], axis=-1)

    m_per_deg_lon = M_PER_DEG_LAT * max(np.cos(mean_lat_rad), 1e-6)
    px_area_m2 = abs(dst_transform.a * dst_transform.e) * (m_per_deg_lon * M_PER_DEG_LAT)
    px_per_segment = max(1, int((TARGET_SUPERPIXEL_M ** 2) / max(px_area_m2, 1e-6)))
    nseg = max(50, int((h * w) / px_per_segment))
    seg = slic(feat, n_segments=nseg, compactness=0.2, start_label=1, channel_axis=-1)

    seg_ids = np.unique(seg)
    seg_conf = np.zeros_like(fused)
    seg_z = np.zeros_like(fused)
    for sid in seg_ids:
        m = (seg == sid) & slope_mask
        if np.any(m):
            seg_conf[seg == sid] = float(np.mean(fused[m]))
            sum_chi = float(np.sum(chi_comb_full[m]))
            sum_df = float(np.sum(df_comb_full[m]))
            if sum_df > 0:
                # CLT approximation: a segment averaging k iid chi-square(df) pixels has
                # mean=sum_df, var=2*sum_df under the no-change null, so this z-score tests
                # whether the segment's aggregate evidence is distinguishable from noise.
                seg_z[seg == sid] = (sum_chi - sum_df) / np.sqrt(2.0 * sum_df)
        # else: segment fully outside slope mask; seg_conf/seg_z stay 0

    # Bonferroni-corrected significance gate across all segments in this incident - a
    # segment must clear this bar (not just rank in the top percentile) to become a
    # candidate, fixing the previous behavior where HIGH_CONF_PERCENTILE always produced
    # a "detection" even in incidents with no statistically real change anywhere.
    n_seg_actual = max(len(seg_ids), 1)
    z_crit = norm.ppf(1 - FWER_ALPHA / n_seg_actual)
    sig_mask = seg_z >= z_crit

    # Fixed absolute confidence cutoffs (e.g. 0.90/0.999) are effectively unreachable in
    # practice: `fused` is a geometric mean of two independently-noisy IR-MAD change
    # confidences, then averaged again per-superpixel, so even genuine landslide change
    # rarely pushes the segment mean anywhere near 0.999 - this was the actual cause of
    # the pipeline returning 0 candidates, not the earlier sign-inversion bug alone.
    # Threshold adaptively per incident instead (matching the workflow's "Otsu per
    # incident, or a fixed percentile" design): take the top HIGH_CONF_PERCENTILE of
    # in-terrain seg_conf values as hysteresis seeds, and the top LOW_CONF_PERCENTILE
    # as the region seeds are allowed to grow into.
    valid_conf = seg_conf[slope_mask]
    if valid_conf.size == 0 or np.max(valid_conf) <= 0:
        mask = np.zeros_like(slope_mask, dtype=bool)
    else:
        high_thr = np.percentile(valid_conf, HIGH_CONF_PERCENTILE)
        low_thr = min(np.percentile(valid_conf, LOW_CONF_PERCENTILE), high_thr)
        mask = apply_hysteresis_threshold(seg_conf, low_thr, high_thr)
    mask = mask & slope_mask & sig_mask
    mask = remove_small_objects(mask, min_size=5)

    lbl = label(mask)
    px_area = px_area_m2
    cand = []
    for rg in regionprops(lbl, intensity_image=fused):
        area = rg.area * px_area
        if area < MIN_BLOB_AREA_M2 or area > MAX_BLOB_AREA_M2:
            continue
        maj = rg.major_axis_length or 1
        minr = rg.minor_axis_length or 1
        elong = maj / max(minr, 1e-6)
        if elong > MAX_ELONGATION:
            continue
        sev = float(rg.mean_intensity)
        score = float(area * sev)
        r0, c0, r1, c1 = rg.bbox
        lon_min, lat_max = dst_transform * (c0, r0)
        lon_max, lat_min = dst_transform * (c1, r1)
        cand.append({'incident_id': inc_id, 'area_m2': float(area), 'elongation': float(elong), 'severity': sev, 'score': score, 'bbox_lonlat': [float(lon_min), float(lat_min), float(lon_max), float(lat_max)], 'bbox_px': [int(r0), int(c0), int(r1), int(c1)]})

    cand = sorted(cand, key=lambda x: x['score'], reverse=True)[:MAX_CANDIDATES_PER_INCIDENT]

    pixel_size_m = max((px_area_m2 ** 0.5), 1e-6)
    pad_px = max(1, int(round(CHIP_PAD_M / pixel_size_m)))
    inc_dir_local = f'/kaggle/working/incident_{inc_id}'
    uploads = []
    for cid, c in enumerate(cand, start=1):
        c['candidate_id'] = cid
        r0, c0, r1, c1 = c.pop('bbox_px')
        pr0, pc0 = max(0, r0 - pad_px), max(0, c0 - pad_px)
        pr1, pc1 = min(h, r1 + pad_px), min(w, c1 + pad_px)
        win = Window(col_off=pc0, row_off=pr0, width=pc1 - pc0, height=pr1 - pr0)
        chip_transform = window_transform(win, dst_transform)

        cand_dir_local = f'{inc_dir_local}/candidate_{cid}'
        os.makedirs(cand_dir_local, exist_ok=True)
        cand_repo_dir = f'{HF_CAND_ROOT}/incident_{inc_id}/candidate_{cid}'
        # Compute the chip geographic bbox in EPSG:4326 from the 10 m analysis grid window.
        lon_min_chip = chip_transform.c
        lat_max_chip = chip_transform.f
        lon_max_chip = chip_transform.c + (pc1 - pc0) * chip_transform.a
        lat_min_chip = chip_transform.f + (pr1 - pr0) * chip_transform.e
        lonlat_bbox = (lon_min_chip, lat_min_chip, lon_max_chip, lat_max_chip)

        # Before/after chips are re-read from the original Planet files at their
        # native resolution (spatial resampling to 10 m was for analysis only).
        # Slope and mask chips stay on the 10 m analysis grid.
        before_rgb_band_indexes = optical_meta['before_rgb_band_indexes'][:n_opt_bands]
        after_rgb_band_indexes = optical_meta['after_rgb_band_indexes'][:n_opt_bands]
        before_chip, before_chip_tf, before_chip_crs = crop_native(before_path, before_rgb_band_indexes, lonlat_bbox)
        after_chip, after_chip_tf, after_chip_crs = crop_native(after_path, after_rgb_band_indexes, lonlat_bbox)

        chip_specs = {
            'before': (before_chip, before_chip_tf, before_chip_crs),
            'after': (after_chip, after_chip_tf, after_chip_crs),
            'slope': (slope[pr0:pr1, pc0:pc1], chip_transform, TARGET_CRS),
            'mask': (mask[pr0:pr1, pc0:pc1].astype(np.float32), chip_transform, TARGET_CRS),
        }
        for chip_type, (chip_arr, chip_tf, chip_crs) in chip_specs.items():
            fn_name = f'incident_{inc_id}_candidate_{cid}_{chip_type}.tif'
            local_path = os.path.join(cand_dir_local, fn_name)
            write_chip(local_path, chip_arr, chip_tf, chip_crs)
            uploads.append((local_path, f'{cand_repo_dir}/{fn_name}'))

    return {
        'incident_id': inc_id, 'status': 'ok', 'n_candidates': len(cand), 'error': '',
        'candidates': cand, 'uploads': uploads,
        'local_dir': inc_dir_local if uploads else None,
    }

incidents_to_process = []
for inc_id in incident_ids:
    inc_cand_prefix = f'{HF_CAND_ROOT}/incident_{inc_id}/'
    if any(f.startswith(inc_cand_prefix) for f in existing_cand_files):
        print(f'Skip incident_{inc_id}: candidates already exist on HF')
        continue
    incidents_to_process.append(inc_id)

records = []
candidate_metadata = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_incident, inc_id): inc_id for inc_id in incidents_to_process}
    for future in as_completed(futures):
        inc_id = futures[future]
        try:
            rec = future.result()
            uploads = rec.pop('uploads', [])
            local_dir = rec.pop('local_dir', None)
            candidates = rec.pop('candidates', [])
            candidate_metadata.extend(candidates)
            for local_path, path_in_repo in uploads:
                pending_ops.append(CommitOperationAdd(path_in_repo=path_in_repo, path_or_fileobj=local_path))
            if local_dir:
                pending_meta.append((inc_id, local_dir))
            if len(pending_ops) >= UPLOAD_BATCH_SIZE:
                flush_pending()
            print(f"incident_{inc_id}: {rec['n_candidates']} candidates")
        except Exception as e:
            rec = {'incident_id': inc_id, 'status': 'failed', 'n_candidates': 0, 'error': str(e)}
            shutil.rmtree(f'/kaggle/working/incident_{inc_id}', ignore_errors=True)
            print(f'incident_{inc_id} failed: {e}')
        records.append(rec)

flush_pending()

status_csv = '/kaggle/working/candidate_status.csv'
new_status = pd.DataFrame(records)
try:
    prior_status_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE,
        revision=HF_REVISION, filename=f'{HF_CAND_ROOT}/candidate_status.csv', token=hf_token)
    prior_status = pd.read_csv(prior_status_path)
    prior_status = prior_status[~prior_status['incident_id'].isin(set(new_status['incident_id']))]
    new_status = pd.concat([prior_status, new_status], ignore_index=True)
except Exception:
    pass
new_status.to_csv(status_csv, index=False)

meta_csv = None
if candidate_metadata:
    meta_rows = [{
        'incident_id': c['incident_id'],
        'candidate_id': c['candidate_id'],
        'area_m2': c['area_m2'],
        'elongation': c['elongation'],
        'severity': c['severity'],
        'score': c['score'],
        'lon_min': c['bbox_lonlat'][0],
        'lat_min': c['bbox_lonlat'][1],
        'lon_max': c['bbox_lonlat'][2],
        'lat_max': c['bbox_lonlat'][3],
    } for c in candidate_metadata]
    meta_csv = '/kaggle/working/candidate_metadata.csv'
    new_meta = pd.DataFrame(meta_rows)
    try:
        prior_meta_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE,
            revision=HF_REVISION, filename=f'{HF_CAND_ROOT}/candidate_metadata.csv', token=hf_token)
        prior_meta = pd.read_csv(prior_meta_path)
        prior_meta = prior_meta[~prior_meta['incident_id'].isin(set(new_meta['incident_id']))]
        new_meta = pd.concat([prior_meta, new_meta], ignore_index=True)
    except Exception:
        pass
    new_meta.to_csv(meta_csv, index=False)

summary_ops = [
    CommitOperationAdd(path_in_repo=f'{HF_CAND_ROOT}/candidate_status.csv', path_or_fileobj=status_csv),
]
if meta_csv is not None:
    summary_ops.append(
        CommitOperationAdd(path_in_repo=f'{HF_CAND_ROOT}/candidate_metadata.csv', path_or_fileobj=meta_csv)
    )

api.create_commit(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
    operations=summary_ops,
    commit_message='Update candidate status and metadata summaries',
)

print('Candidate detection pass complete')
